In [1]:
# """
# Train a Vision Transformer (ViT) from scratch (randomly initialized, NOT
# pretrained) on Imagenette, where image paths and labels come from a CSV
# file in /Data — and sweep over ViT hyperparameters (patch size, embedding
# dimension, attention heads, hidden layers), changing only ONE parameter
# at a time relative to a fixed baseline configuration.
 
# For each combination tried, this script:
#   - trains a fresh ViT from scratch for EPOCHS epochs
#   - saves an accuracy-over-time plot for that combination, with a title
#     describing exactly which hyperparameters were used
#   - records the best (maximum) validation accuracy reached
 
# At the end, it writes a summary CSV with one row per combination, so you
# can compare all the max validation accuracies side by side.
 
# Needs: torch, torchvision, matplotlib, tqdm (no Hugging Face libraries).


In [2]:
import csv
import gc
import os
import random
 
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torch.utils.checkpoint import checkpoint
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

In [3]:
%cd ..


%pwd

/users/gcb25209


'/users/gcb25209'

In [4]:
# ---------------------------------------------------------------------------
# 0. CONFIG — adjust these to match your CSV
# ---------------------------------------------------------------------------

DATA_DIR = "/users/gcb25209/Data/imagenette2"
CSV_PATH = "/users/gcb25209/Data/imagenette2/noisy_imagenette.csv"   # <-- change to your actual CSV filename
HAS_HEADER = True
VAL_FRACTION = 0.2              # 20% of rows held out for validation
RANDOM_SEED = 42

In [ ]:
# ---------------------------------------------------------------------------
# 0b. ADVERSARIAL DATASETS: CONFIG AND HELPERS
# ---------------------------------------------------------------------------
# Each adversarial dataset is a folder of class sub-folders (tench/,
# english_springer/, ...) containing .png files. Labels come from the FOLDER
# name, because some files (e.g. ILSVRC2012_val_*.png) have no class prefix.
import re

ADV1_DIR = "/users/gcb25209/Data/adversarial_1"   # <-- CHANGE to the first adversarial dataset
ADV2_DIR = "/users/gcb25209/Data/adversarial_2"   # <-- CHANGE to the second adversarial dataset
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}

# Folder name -> class id used in the CSV (the standard Imagenette mapping).
FOLDER_TO_WNID = {
    "tench": "n01440764",
    "english_springer": "n02102040",
    "cassette_player": "n02979186",
    "chain_saw": "n03000684",
    "church": "n03028079",
    "french_horn": "n03394916",
    "garbage_truck": "n03417042",
    "gas_pump": "n03425413",
    "golf_ball": "n03445777",
    "parachute": "n03888257",
}


def file_stem(path):
    """'train/n01440764/n01440764_6246.JPEG' -> 'n01440764_6246'"""
    return os.path.splitext(os.path.basename(path))[0]


def index_adversarial_images(adv_dir):
    """Return {file_stem: (relative_path, wnid)} for every image under adv_dir."""
    index = {}
    for folder, wnid in FOLDER_TO_WNID.items():
        folder_path = os.path.join(adv_dir, folder)
        if not os.path.isdir(folder_path):
            raise FileNotFoundError(f"Expected class folder not found: {folder_path}")
        for fname in sorted(os.listdir(folder_path)):
            stem, ext = os.path.splitext(fname)
            if ext.lower() not in IMAGE_EXTENSIONS:
                continue
            # Sanity check: where the file name carries a class id, it must
            # agree with the folder it sits in.
            m = re.match(r"(n\d{8})_", stem)
            if m and m.group(1) != wnid:
                raise ValueError(f"{fname} is in '{folder}' but has class id {m.group(1)} (expected {wnid})")
            if stem in index:
                raise ValueError(f"Duplicate image name {stem} in {adv_dir}")
            index[stem] = (os.path.join(folder, fname), wnid)
    return index


In [5]:
# ---------------------------------------------------------------------------
# 1. READ THE CSV AND BUILD (path, label) PAIRS
# ---------------------------------------------------------------------------
def load_csv_rows(csv_path, has_header):
#    """Read the CSV and return a list of (image_path, label_str) tuples."""
    rows = []
    with open(csv_path, newline="") as f:
        reader = csv.reader(f)
        if has_header:
            next(reader)  # skip header row
        for row in reader:
            image_path, label = row[0], row[1]
            rows.append((image_path, label))
    return rows


all_rows = load_csv_rows(CSV_PATH, HAS_HEADER)

# Build a sorted list of unique class names, then a name -> integer index
# mapping (models need integer labels, not strings).
class_names = sorted({label for _, label in all_rows})
label_to_index = {name: i for i, name in enumerate(class_names)}
num_labels = len(class_names)
print(f"Found {num_labels} classes: {class_names}")

assert set(FOLDER_TO_WNID.values()) == set(label_to_index), \
    "FOLDER_TO_WNID doesn't match the classes found in the CSV"

# ---------------------------------------------------------------------------
# MATCHED SPLIT: the clean validation set, adversarial set 1 and adversarial
# set 2 must contain the SAME images (identified by file name).
#
# A plain random split would put only ~VAL_FRACTION of the images that exist
# in all three places into validation, and no choice of seed fixes that. So
# instead: find the images present in the CSV AND in both adversarial sets,
# shuffle them with RANDOM_SEED, and fill the validation set from them first
# (up to the usual VAL_FRACTION size). Everything else is training data.
# ---------------------------------------------------------------------------
csv_stems = [file_stem(path) for path, _ in all_rows]
assert len(set(csv_stems)) == len(csv_stems), "file names in the CSV are not unique"

adv1_index = index_adversarial_images(ADV1_DIR)
adv2_index = index_adversarial_images(ADV2_DIR)

common_stems = set(csv_stems) & adv1_index.keys() & adv2_index.keys()
print(f"CSV: {len(all_rows)} images | adv1: {len(adv1_index)} | adv2: {len(adv2_index)} "
      f"| present in all three: {len(common_stems)}")
if not common_stems:
    raise ValueError(
        "No image names are shared between the CSV and both adversarial sets. "
        f"Example CSV name: {csv_stems[0]!r}; example adv1 name: {next(iter(adv1_index), None)!r}"
    )

random.seed(RANDOM_SEED)
random.shuffle(all_rows)

matched_rows = [r for r in all_rows if file_stem(r[0]) in common_stems]
unmatched_rows = [r for r in all_rows if file_stem(r[0]) not in common_stems]

# Same validation size as before (20%), unless fewer matched images exist.
target_val_size = len(all_rows) - int(len(all_rows) * (1 - VAL_FRACTION))
n_val = min(target_val_size, len(matched_rows))

val_rows = matched_rows[:n_val]
train_rows = matched_rows[n_val:] + unmatched_rows
random.shuffle(train_rows)
print(f"Train rows: {len(train_rows)} | Val rows: {len(val_rows)}"
      + ("" if n_val == target_val_size else f"  (smaller than the usual {target_val_size}: not enough matched images)"))

# The clean val labels come from the CSV and the adversarial labels from the
# folder names; for the same image these should agree. Report if they don't.
label_mismatches = sum(
    1 for path, label in val_rows
    if adv1_index[file_stem(path)][1] != label or adv2_index[file_stem(path)][1] != label
)
print(f"Val images where the CSV label differs from an adversarial folder label: {label_mismatches}")




Found 10 classes: ['n01440764', 'n02102040', 'n02979186', 'n03000684', 'n03028079', 'n03394916', 'n03417042', 'n03425413', 'n03445777', 'n03888257']
Train rows: 10715 | Val rows: 2679


In [6]:
# ---------------------------------------------------------------------------
# 2. A CUSTOM DATASET THAT READS IMAGES USING THE CSV ROWS
# ---------------------------------------------------------------------------
IMAGE_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),                 # PIL image -> [0,1] tensor, shape (C,H,W)
    transforms.Normalize(mean=MEAN, std=STD),
])


class CsvImageDataset(Dataset):
#    """
#    A minimal PyTorch Dataset that, for each CSV row, loads the image at
#    `data_dir/path` from disk and returns (transformed_image, label_index).
#    """

    def __init__(self, rows, data_dir, label_to_index, transform):
        self.rows = rows
        self.data_dir = data_dir
        self.label_to_index = label_to_index
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        image_path, label_str = self.rows[idx]
        full_path = os.path.join(self.data_dir, image_path)

        image = Image.open(full_path).convert("RGB")  # force 3 channels
        image = self.transform(image)

        label = self.label_to_index[label_str]
        return image, label


train_data = CsvImageDataset(train_rows, DATA_DIR, label_to_index, transform)
val_data = CsvImageDataset(val_rows, DATA_DIR, label_to_index, transform)

# These dataloaders are shared by every hyperparameter combination below —
# the data itself doesn't change, only the model architecture and training
# hyperparameters do.
BATCH_SIZE = 64
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.enabled = False


/opt/software/miniforge/python-3.12.10/25.3.0/lib/python3.12/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 2 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [ ]:
# ---------------------------------------------------------------------------
# 2b. ADVERSARIAL VALIDATION LOADERS
# ---------------------------------------------------------------------------
# Built from the SAME images as val_rows, in the same order: item i of the
# clean, adv1 and adv2 loaders is the same underlying image.
adv1_rows = [adv1_index[file_stem(path)] for path, _ in val_rows]
adv2_rows = [adv2_index[file_stem(path)] for path, _ in val_rows]
assert [file_stem(p) for p, _ in adv1_rows] == [file_stem(p) for p, _ in val_rows]
assert [file_stem(p) for p, _ in adv2_rows] == [file_stem(p) for p, _ in val_rows]

adv1_data = CsvImageDataset(adv1_rows, ADV1_DIR, label_to_index, transform)
adv2_data = CsvImageDataset(adv2_rows, ADV2_DIR, label_to_index, transform)
adv1_loader = DataLoader(adv1_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
adv2_loader = DataLoader(adv2_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Every epoch, the model is scored on each of these in turn.
val_loaders = {"clean": val_loader, "adv1": adv1_loader, "adv2": adv2_loader}
print({name: len(loader.dataset) for name, loader in val_loaders.items()})


In [7]:
# ---------------------------------------------------------------------------
# 3. VISION TRANSFORMER MODEL DEFINITION
# ---------------------------------------------------------------------------
class PatchEmbedding(nn.Module):
#    """
#    Splits an image into non-overlapping patch_size x patch_size patches
#    and linearly projects each one into an embed_dim-length vector, via a
#    Conv2d whose kernel size AND stride both equal patch_size (so each
#    "convolution" touches exactly one non-overlapping patch).
#    """
 
    def __init__(self, patch_size, num_channels, embed_dim):
        super().__init__()
        self.projection = nn.Conv2d(
            num_channels, embed_dim, kernel_size=patch_size, stride=patch_size
        )
 
    def forward(self, x):
        x = self.projection(x)   # (batch, embed_dim, H/patch, W/patch)
        x = x.flatten(2)         # (batch, embed_dim, num_patches)
        x = x.transpose(1, 2)    # (batch, num_patches, embed_dim)
        return x
 
 
class VisionTransformer(nn.Module):
    def __init__(
        self,
        image_size,
        patch_size,
        num_channels,
        embed_dim,
        num_heads,
        num_layers,
        num_classes,
        mlp_ratio=4.0,
        dropout=0.1,
        use_checkpoint=False,
    ):
        super().__init__()
        self.use_checkpoint = use_checkpoint
        num_patches = (image_size // patch_size) ** 2
 
        self.patch_embed = PatchEmbedding(patch_size, num_channels, embed_dim)
 
        # A learnable "classification token" prepended to the patch
        # sequence; after self-attention mixes information between all
        # patches, this token's final embedding represents the whole image.
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
 
        # Learnable position embeddings — one per patch, plus one for the
        # [CLS] token — since self-attention has no built-in notion of
        # where each patch sits in the image.
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)
 
        # `norm_first=True` applies LayerNorm BEFORE attention / feed-forward
        # in each block ("pre-norm") instead of after. Pre-norm is what most
        # ViT implementations use because post-norm (PyTorch's default)
        # becomes unstable as models get wider/deeper, which showed up as
        # flat, non-learning training at embed_dim=768.
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers, enable_nested_tensor=False
        )
 
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
 
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
 
    def forward_features(self, x):
        batch_size = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.dropout(x)
        x = self.encoder(x)
        x = self.norm(x)
        return x[:, 0]
 
    def forward(self, x):
        cls_embedding = self.forward_features(x)
        return self.head(cls_embedding)
 
 

In [8]:
# ---------------------------------------------------------------------------
# 4. TRAIN-ONE-CONFIGURATION FUNCTION
# ---------------------------------------------------------------------------
# Everything needed to build, train, and evaluate a ViT for ONE particular
# choice of (patch_size, embed_dim, num_heads, num_layers) is wrapped in
# this function, so the hyperparameter sweep in step 5 can just call it
# once per combination.
EPOCHS = 15
LEARNING_RATE = 1e-4    # lowered from 3e-4: wide models (embed_dim=768) failed to learn at the higher rate
WARMUP_EPOCHS = 5       # lengthened from 2: wide models are most sensitive to the earliest steps
WEIGHT_DECAY = 0.05
MAX_GRAD_NORM = 1.0
MLP_RATIO = 4.0
DROPOUT = 0.1


def train_one_configuration(patch_size, embed_dim, num_heads, num_layers, title, filename_tag,
                            accum_steps=1):
    """
    Builds a fresh, randomly-initialized ViT with the given hyperparameters,
    trains it from scratch for EPOCHS epochs, saves an accuracy-over-time
    plot titled `title`, and returns a dict with the maximum accuracy reached on each
    validation set (clean, adv1, adv2).

    accum_steps: each batch of BATCH_SIZE images is processed in this many
    smaller chunks, one after another, to reduce peak GPU memory. The
    effective batch size, and the weight update per batch, stay the same
    (see the training loop below), so the experiment is unchanged.
    """
    assert IMAGE_SIZE % patch_size == 0, "IMAGE_SIZE must be divisible by patch_size"
    assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"

    model = VisionTransformer(
        image_size=IMAGE_SIZE,
        patch_size=patch_size,
        num_channels=3,
        embed_dim=embed_dim,
        num_heads=num_heads,
        num_layers=num_layers,
        num_classes=num_labels,
        mlp_ratio=MLP_RATIO,
        dropout=DROPOUT,
    ).to(device)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"\n=== {title} ({num_params:,} parameters) ===")

    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()

    # Linear warmup + cosine decay, same rationale as before: jumping
    # straight to the target learning rate destabilizes Transformers early
    # in training, so ramp up first, then ease back down.
    steps_per_epoch = len(train_loader)
    warmup_steps = WARMUP_EPOCHS * steps_per_epoch
    total_steps = EPOCHS * steps_per_epoch

    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return current_step / max(1, warmup_steps)
        progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + torch.cos(torch.tensor(progress * 3.14159265)).item())

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

    train_accuracies = []
    # One accuracy-over-time list per validation set: clean, adv1, adv2.
    val_accuracies = {name: [] for name in val_loaders}

    def run_validation(loader):
        model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, targets in loader:
                images, targets = images.to(device), targets.to(device)
                logits = model(images)
                loss = loss_fn(logits, targets)
                total_loss += loss.item() * images.size(0)
                predictions = logits.argmax(dim=1)
                correct += (predictions == targets).sum().item()
                total += targets.size(0)
        return total_loss / total, correct / total

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        progress_bar = tqdm(
            train_loader, desc=f"[{filename_tag}] Epoch {epoch + 1}/{EPOCHS}", unit="batch"
        )

        samples_seen = 0
        for images, targets in progress_bar:
            images, targets = images.to(device), targets.to(device)

            optimizer.zero_grad()

            # GRADIENT ACCUMULATION: instead of pushing all BATCH_SIZE images
            # through the model at once (which keeps every layer's
            # activations in memory until backward), process them in
            # `accum_steps` smaller chunks. Each chunk's backward() pass adds
            # to the gradients and then frees that chunk's activations, so
            # peak memory drops roughly in proportion to accum_steps.
            #
            # Each chunk's loss is weighted by its share of the batch, so the
            # summed gradients equal the gradient of the mean loss over the
            # whole batch, i.e. the same update as one big batch. (A ViT has
            # no batch-dependent layers like BatchNorm, so this is
            # equivalent; only the dropout random numbers differ.)
            batch_loss = 0.0
            batch_correct = 0
            for chunk_images, chunk_targets in zip(
                images.chunk(accum_steps), targets.chunk(accum_steps)
            ):
                chunk_logits = model(chunk_images)
                chunk_loss = loss_fn(chunk_logits, chunk_targets)
                (chunk_loss * chunk_images.size(0) / images.size(0)).backward()

                batch_loss += chunk_loss.item() * chunk_images.size(0)
                batch_correct += (chunk_logits.argmax(dim=1) == chunk_targets).sum().item()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)

            optimizer.step()
            scheduler.step()

            running_loss += batch_loss
            samples_seen += images.size(0)

            correct_train += batch_correct
            total_train += targets.size(0)

            progress_bar.set_postfix(
                loss=running_loss / samples_seen,
                lr=optimizer.param_groups[0]["lr"],
            )

        train_loss = running_loss / len(train_data)
        train_acc = correct_train / total_train
        # Test the clean validation set first, then swap in each adversarial
        # set in turn. Same model weights for all three within this epoch.
        val_results = {name: run_validation(loader) for name, loader in val_loaders.items()}

        train_accuracies.append(train_acc)
        for name, (_, acc) in val_results.items():
            val_accuracies[name].append(acc)

        print(
            f"[{filename_tag}] Epoch {epoch + 1}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            + " | ".join(f"{name}_loss={l:.4f} {name}_acc={a:.4f}"
                         for name, (l, a) in val_results.items())
        )

    # --- Save the accuracy-over-time plot for this combination. ---
    epochs_range = range(1, EPOCHS + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs_range, train_accuracies, marker="o", label="Train accuracy")
    plt.plot(epochs_range, val_accuracies["clean"], marker="o", label="Validation accuracy (clean)")
    plt.plot(epochs_range, val_accuracies["adv1"], marker="s", label="Validation accuracy (adversarial 1)")
    plt.plot(epochs_range, val_accuracies["adv2"], marker="^", label="Validation accuracy (adversarial 2)")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(True)

    plot_path = f"./accuracy_{filename_tag}.png"
    plt.savefig(plot_path)
    plt.close()  # close (rather than show) so the sweep doesn't pause for each plot
    print(f"Saved accuracy plot to {plot_path}")

    max_accs = {
        "max_val_accuracy": max(val_accuracies["clean"]),
        "max_adv1_accuracy": max(val_accuracies["adv1"]),
        "max_adv2_accuracy": max(val_accuracies["adv2"]),
    }

    # --- Free this run's GPU memory before the next combination starts. ---
    # Without this, the model/optimizer/scheduler from THIS run stay
    # resident on the GPU while the next one is built, and over enough
    # combinations that steadily accumulates until CUDA runs out of
    # memory. Deleting the references and clearing PyTorch's cache
    # reclaims that memory for the next run.
    del model, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return max_accs


 


In [9]:
# ---------------------------------------------------------------------------
# 5. HYPERPARAMETER SWEEP — FULL GRID
# ---------------------------------------------------------------------------
# Every combination of every value below is tried: 3 * 3 * 3 * 3 = 81 full
# training runs in total. This is exhaustive (it directly compares every
# possible combination, not just single-parameter effects around one
# baseline), but is also 81x the compute of training just one
# configuration — expect this to take a long time. Lower EPOCHS in step 4
# for the sweep itself if you want a faster first pass.
import itertools
 
PARAM_GRIDS = {
    "patch_size": [8, 16, 32],
    "embed_dim": [192, 384, 768],
    "num_heads": [3, 6, 12],
    "num_layers": [3, 6, 12],
}
 
# `itertools.product` generates every combination of the 4 lists above,
# i.e. the full Cartesian product / grid.
combinations_to_run = [
    dict(zip(PARAM_GRIDS.keys(), values))
    for values in itertools.product(*PARAM_GRIDS.values())
]
 
# ---------------------------------------------------------------------------
# QUICK TEST SWITCH
# ---------------------------------------------------------------------------
# Set QUICK_TEST = True to run ONLY the single combination that failed to
# learn before (embed_dim=768), for a short run, to check whether the new
# learning rate / warmup / pre-norm settings fix it BEFORE committing to
# all 81 runs. Set it back to False for the full sweep.
#
# 10 epochs is used rather than fewer because the first 5 are warmup —
# a shorter run would never leave the warmup phase.
QUICK_TEST = False
QUICK_TEST_COMBO = {"patch_size": 16, "embed_dim": 768, "num_heads": 12, "num_layers": 6}
QUICK_TEST_EPOCHS = 10
 
if QUICK_TEST:
    combinations_to_run = [QUICK_TEST_COMBO]
    EPOCHS = QUICK_TEST_EPOCHS   # train_one_configuration reads this global
    print(f"QUICK_TEST is ON: running only {QUICK_TEST_COMBO} for {EPOCHS} epochs")

START_AT = 46

print(f"\nRunning {len(combinations_to_run)} combinations (full grid sweep)...")
 
# Quick-test results go to a separate file so they never mix with the real sweep.
# New file names: the columns changed, so they must not be appended to the old CSVs.
summary_path = "./quick_test_results_adversarial.csv" if QUICK_TEST else "./hyperparameter_sweep_results_adversarial.csv"
fieldnames = ["patch_size", "embed_dim", "num_heads", "num_layers",
              "max_val_accuracy", "max_adv1_accuracy", "max_adv2_accuracy"]
 
# Results are appended one row at a time as each combination finishes (see
# the loop below), so a crash never loses completed work.
#
# RESUME SUPPORT: if the CSV already exists (e.g. from an earlier run that
# crashed partway through), its rows are KEPT and those combinations are
# skipped, so re-running this script picks up where it left off instead of
# starting over (and instead of overwriting the old results). To start
# completely fresh, delete the CSV file first.
completed = set()
if os.path.exists(summary_path):
    with open(summary_path, newline="") as f:
        for row in csv.DictReader(f):
            completed.add((int(row["patch_size"]), int(row["embed_dim"]),
                           int(row["num_heads"]), int(row["num_layers"])))
    print(f"Found {len(completed)} completed combinations in {summary_path}; they will be skipped.")
else:
    with open(summary_path, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=fieldnames).writeheader()
 
# Combinations that fail for memory reasons even after retrying are logged
# here, so they're not silently lost.
skipped_path = "./skipped_combinations.csv"
 
# On a memory error, the combination is retried with progressively more
# aggressive memory saving, in this order. Each entry is
# (accum_steps, use_checkpoint):
#   accum_steps    = number of chunks each batch is split into (1 = no splitting)
#   use_checkpoint = recompute activations during backward instead of storing them
# All settings train equivalently; the later ones are just slower.
MEMORY_ATTEMPTS = [
    (1, False),
    (2, False),
    (4, False),
    (8, False),
    (8, True),    # accumulation alone wasn't enough: add checkpointing
    (32, True),   # last resort: 1 image per chunk (batch of 32 / 32)
]
 
 
def is_memory_error(exc):
    """True if this exception looks like the GPU ran out of memory."""
    if isinstance(exc, torch.cuda.OutOfMemoryError):
        return True
    # When memory is nearly exhausted, cuDNN (the GPU convolution library)
    # often can't allocate its workspace and raises a plain RuntimeError
    # instead of an OutOfMemoryError. This is the message it produces.
    message = str(exc).lower()
    return "out of memory" in message or "unable to find an engine" in message
 
 
results = []
for i, combo in enumerate(combinations_to_run, start=1):

    if i < START_AT:
        continue
        
    patch_size = combo["patch_size"]
    embed_dim = combo["embed_dim"]
    num_heads = combo["num_heads"]
    num_layers = combo["num_layers"]
 
    if (patch_size, embed_dim, num_heads, num_layers) in completed:
        print(f"Skipping combo {i}/{len(combinations_to_run)}: already completed")
        continue
 
    # Skip combinations that aren't architecturally valid: each attention
    # head needs an equal, whole-number share of the embedding dimension.
    if embed_dim % num_heads != 0:
        print(
            f"Skipping combo {i}/{len(combinations_to_run)}: "
            f"embed_dim={embed_dim} not divisible by num_heads={num_heads}"
        )
        continue
 
    filename_tag = f"p{patch_size}_e{embed_dim}_h{num_heads}_l{num_layers}"
    title = (
        f"Accuracy over time ({i}/{len(combinations_to_run)})\n"
        f"patch_size={patch_size}, embed_dim={embed_dim}, "
        f"num_heads={num_heads}, num_layers={num_layers}"
    )
 
    # Try the normal single-pass batch first. If that runs out of memory,
    # retry the SAME combination with progressively more memory saving (see
    # MEMORY_ATTEMPTS above). Training is equivalent each time; later
    # attempts just use less memory and run more slowly.
    failed_message = None
    for attempt_number, (accum_steps, use_checkpoint) in enumerate(MEMORY_ATTEMPTS, start=1):
        failed_message = None
        try:
            max_accs = train_one_configuration(
                patch_size=patch_size,
                embed_dim=embed_dim,
                num_heads=num_heads,
                num_layers=num_layers,
                title=title,
                filename_tag=filename_tag,
                accum_steps=accum_steps,
                
            )
        except RuntimeError as e:      # OutOfMemoryError is a subclass of RuntimeError
            if not is_memory_error(e):
                raise                  # a genuine bug: don't hide it
            failed_message = str(e).splitlines()[0]
 
        if failed_message is None:
            break
 
        # Free the failed attempt's memory (outside the except block, so it
        # is actually released) before retrying or moving on.
        gc.collect()
        torch.cuda.empty_cache()
        if attempt_number < len(MEMORY_ATTEMPTS):
            print(f"[{filename_tag}] out of memory at accum_steps={accum_steps}, "
                  f"checkpointing={use_checkpoint}; retrying with more memory saving")
 
    if failed_message is not None:
        # Memory is freed HERE, after the `except` block has ended, rather
        # than inside it. While an exception is being handled, Python keeps
        # the failed run's model, optimizer and activations alive (the
        # traceback references them), so calling empty_cache() inside the
        # `except` block frees almost nothing and the next combination
        # inherits a nearly-full GPU.
        gc.collect()
        torch.cuda.empty_cache()
        print(f"Skipping combo {i}/{len(combinations_to_run)} ({filename_tag}): "
              f"GPU memory / cuDNN error ({failed_message})")
        with open(skipped_path, "a", newline="") as f:
            csv.writer(f).writerow([patch_size, embed_dim, num_heads, num_layers, failed_message])
        continue
 
    result_row = {
        "patch_size": patch_size,
        "embed_dim": embed_dim,
        "num_heads": num_heads,
        "num_layers": num_layers,
        **max_accs,
    }
    results.append(result_row)
 
    # Append immediately, so this result is saved even if a later
    # combination crashes the sweep.
    with open(summary_path, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=fieldnames).writerow(result_row)


Running 81 combinations (full grid sweep)...
Found 45 completed combinations in ./hyperparameter_sweep_results.csv; they will be skipped.

=== Accuracy over time (46/81)
patch_size=16, embed_dim=768, num_heads=3, num_layers=3 (22,015,498 parameters) ===


[p16_e768_h3_l3] Epoch 1/15: 100%|██████████| 168/168 [01:48<00:00,  1.55batch/s, loss=2.06, lr=2e-5]   


[p16_e768_h3_l3] Epoch 1/15 | train_loss=2.0632 | train_acc=0.2619 | val_loss=1.8403 | val_acc=0.3520


[p16_e768_h3_l3] Epoch 2/15: 100%|██████████| 168/168 [01:20<00:00,  2.07batch/s, loss=1.77, lr=4e-5]   


[p16_e768_h3_l3] Epoch 2/15 | train_loss=1.7652 | train_acc=0.3851 | val_loss=1.7066 | val_acc=0.4151


[p16_e768_h3_l3] Epoch 3/15: 100%|██████████| 168/168 [01:19<00:00,  2.12batch/s, loss=1.55, lr=6e-5]   


... (1,071 outputs left out)

[p32_e768_h12_l12] Epoch 14/15: 100%|██████████| 168/168 [01:21<00:00,  2.06batch/s, loss=0.0645, lr=2.45e-6]


[p32_e768_h12_l12] Epoch 14/15 | train_loss=0.0645 | train_acc=0.9863 | val_loss=2.0555 | val_acc=0.5767


[p32_e768_h12_l12] Epoch 15/15: 100%|██████████| 168/168 [01:21<00:00,  2.07batch/s, loss=0.0467, lr=0]       


[p32_e768_h12_l12] Epoch 15/15 | train_loss=0.0467 | train_acc=0.9907 | val_loss=2.0590 | val_acc=0.5831
Saved accuracy plot to ./accuracy_p32_e768_h12_l12.png


In [ ]:
#---------------------------------------------------------------------------
# 6. SUMMARY
# ---------------------------------------------------------------------------
# Every row was already appended to summary_path as its combination
# finished (see step 5), so there's nothing left to write here — just
# report what was saved.
print(f"\nSaved {len(results)} results to {summary_path}")
for row in results:
    print(row)

In [ ]:
import sys, gc, torch
sys.last_traceback = sys.last_value = sys.last_type = None
gc.collect()
torch.cuda.empty_cache()

In [ ]:
nvidia-smi
